<a href="https://colab.research.google.com/github/aimeerim1/mental-health-project/blob/main/MENtalbert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers torch scikit-learn

In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import numpy as np

# MentalBERT tokenizer
tokenizer_bert = AutoTokenizer.from_pretrained("distilbert-base-uncased")

print("Tokenizer yüklendi!")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer yüklendi!


In [6]:
import pandas as pd
import re

df = pd.read_csv("Combined Data.csv")
df = df.drop(columns=["Unnamed: 0"])

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = text.strip()
    return text

df = df.dropna()
df["label"] = df["status"].apply(lambda x: 0 if x == "Normal" else 1)
df["clean_statement"] = df["statement"].apply(clean_text)

print("Veri hazır:", df.shape)

Veri hazır: (52681, 4)


In [4]:
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    df["clean_statement"].values,
    df["label"].values,
    test_size=0.2,
    random_state=42
)

class MentalHealthDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = MentalHealthDataset(X_train_b, y_train_b, tokenizer_bert)
test_dataset = MentalHealthDataset(X_test_b, y_test_b, tokenizer_bert)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

print("Dataset hazır!")

NameError: name 'df' is not defined

In [3]:
import torch
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification

# GPU varsa kullan
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Cihaz:", device)

# Modeli yükle
model_bert = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)
model_bert = model_bert.to(device)

# Optimizer
optimizer = AdamW(model_bert.parameters(), lr=2e-5)

# Eğitim döngüsü
EPOCHS = 3

for epoch in range(EPOCHS):
    model_bert.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model_bert(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss/len(train_loader):.4f}")

print("Eğitim tamamlandı!")

Cihaz: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


NameError: name 'train_loader' is not defined

In [3]:
# ============ 1. KÜTÜPHANELER ============
import pandas as pd
import re
import numpy as np
import torch
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# ============ 2. VERİ YÜKLE ============
df = pd.read_csv("Combined Data.csv")
df = df.drop(columns=["Unnamed: 0"])
df = df.dropna()

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    return text.strip()

df["label"] = df["status"].apply(lambda x: 0 if x == "Normal" else 1)
df["clean_statement"] = df["statement"].apply(clean_text)
print("Veri hazır:", df.shape)

# ============ 3. TOKENIZER ============
tokenizer_bert = AutoTokenizer.from_pretrained("distilbert-base-uncased")
print("Tokenizer yüklendi!")

# ============ 4. DATASET ============
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    df["clean_statement"].values,
    df["label"].values,
    test_size=0.2,
    random_state=42
)

class MentalHealthDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = MentalHealthDataset(X_train_b, y_train_b, tokenizer_bert)
test_dataset = MentalHealthDataset(X_test_b, y_test_b, tokenizer_bert)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)
print("Dataset hazır! Train:", len(train_dataset), "Test:", len(test_dataset))

# ============ 5. MODEL EĞİTİM ============
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Cihaz:", device)

model_bert = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)
model_bert = model_bert.to(device)
optimizer = AdamW(model_bert.parameters(), lr=2e-5)

EPOCHS = 3
for epoch in range(EPOCHS):
    model_bert.train()
    total_loss = 0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        optimizer.zero_grad()
        outputs = model_bert(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {total_loss/len(train_loader):.4f}")

print("Eğitim tamamlandı!")

# ============ 6. DEĞERLENDİRME ============
model_bert.eval()
all_preds, all_probs, all_labels = [], [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        outputs = model_bert(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=1)[:, 1]
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Accuracy :", accuracy_score(all_labels, all_preds))
print("F1 Score :", f1_score(all_labels, all_preds))
print("ROC-AUC  :", roc_auc_score(all_labels, all_probs))

Veri hazır: (52681, 4)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

Tokenizer yüklendi!
Dataset hazır! Train: 42144 Test: 10537
Cihaz: cuda


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/3 - Loss: 0.1454
Epoch 2/3 - Loss: 0.0733
Epoch 3/3 - Loss: 0.0371
Eğitim tamamlandı!
Accuracy : 0.9632722786371832
F1 Score : 0.9731641356355315
ROC-AUC  : 0.9925297526103631
